# 3.9 · 数据泄漏 / Data Leakage ⭐

> **课程定位 / Where this fits**
> **Part 3 第 9 课, 全 Part 最重要的一课**。前 8 课每节都在提"防泄漏"; 这一课集中火力。**数据泄漏 = 让模型偷看了它本不该知道的信息 → 训练/验证分数虚高 → 生产崩盘**。这是**实习生和资深 DS 最大的区别之一**, 也是模型"实验室神, 上线废"的头号元凶。
> Leakage = the model peeks at info it shouldn't have → inflated offline scores → production collapse. The #1 cause of "great in the lab, useless in prod".

> 💡 **面试相关 / Interview-relevant**
> - "什么是数据泄漏, 举例" ★★★★★（高频, 答得好显著加分）
> - "怎么发现/防止泄漏" ★★★★★
> - "你遇到过 train 95% test 60% 的情况吗" ★★★★（泄漏 or 过拟合的诊断）

---

## 学习目标 / Learning Objectives
1. 识别泄漏的**所有形态**：目标泄漏 / 预处理泄漏 / 时间泄漏 / 分组泄漏 / 重复泄漏。
2. 用代码**制造并修复**每种泄漏, 看分数怎么虚高又回落。
3. 建立**泄漏侦测**直觉（分数好得不真实 = 警报）。
4. 用 Pipeline + 正确划分构建**结构性防御**。

## 目录 / TOC
1. [泄漏的本质 + 五种形态 ⭐](#1)
2. [目标泄漏：最隐蔽的杀手](#2)
3. [预处理泄漏：fit 在全数据](#3)
4. [时间泄漏：用了未来](#4)
5. [分组泄漏：同一实体跨 train/test](#5)
6. [重复样本泄漏](#6)
7. [泄漏侦测清单 ⭐](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 泄漏的本质 + 五种形态 ⭐ / The Essence & Five Forms

**本质一句话**：**任何在"预测时刻不可能拿到"的信息进入了训练**。

| 形态 | 机制 | 典型例子 |
|---|---|---|
| **目标泄漏** | 特征里藏着目标信息 | 用"是否已退款"预测"是否欺诈"(退款是欺诈的结果) |
| **预处理泄漏** | scaler/imputer/encoder 在全数据上 fit | 用全数据均值填补 (含 test) |
| **时间泄漏** | 用了预测时刻之后的数据 | 用未来 7 天均值预测今天 |
| **分组泄漏** | 同一实体的样本分散在 train 和 test | 同一病人的多次就诊分到两边 |
| **重复泄漏** | 重复样本跨 train/test | 去重前划分, 同一行两边都有 |

**统一信号**：**离线分数好得不真实**。CV 0.99 / test 0.98 别高兴——先怀疑泄漏。
The unified symptom: offline scores that are too good to be true. CV 0.99? Suspect leakage before celebrating.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 构造一个"信贷违约"数据集 / synthetic credit default dataset
n = 4000
income = rng.lognormal(10, 0.5, n)
age = rng.uniform(22, 70, n)
debt = rng.lognormal(8, 0.8, n)
# 真实违约: 受 debt/income 影响 / true default depends on debt-to-income
logit = -3 + 2*(debt/income) + rng.normal(0, 0.5, n)
default = (1/(1+np.exp(-logit)) > 0.5).astype(int)
df = pd.DataFrame({"income":income,"age":age,"debt":debt,"default":default})
print(f"违约率: {df.default.mean():.1%}")


<a id="2"></a>
## 2. 目标泄漏：最隐蔽的杀手 / Target Leakage

**最难发现, 因为它伪装成"好特征"**。某列和目标高度相关——但它其实是目标的**结果**而非**原因**, 预测时根本拿不到。


In [ ]:
# 加一个泄漏特征: "催收次数" — 它是违约的结果, 不是原因!
# Leaky feature: collection_calls is a CONSEQUENCE of default, unavailable at prediction time
df_leak = df.copy()
df_leak["collection_calls"] = df_leak["default"] * rng.poisson(8, n) + rng.poisson(0.2, n)
# 违约者被催收多次, 没违约的几乎不催 → 和目标几乎完美相关

print(f"collection_calls 与 default 相关: {df_leak['collection_calls'].corr(df_leak['default']):.3f}")
print("看起来是超强特征! 但它是违约'之后'才产生的 — 申请时根本没有这个值\n")

X_clean = df_leak[["income","age","debt"]]
X_leak = df_leak[["income","age","debt","collection_calls"]]
y = df_leak["default"]

acc_clean = cross_val_score(RandomForestClassifier(random_state=0), X_clean, y, cv=5).mean()
acc_leak = cross_val_score(RandomForestClassifier(random_state=0), X_leak, y, cv=5).mean()
print(f"干净特征:        CV 准确率 = {acc_clean:.1%}")
print(f"+ 泄漏特征:      CV 准确率 = {acc_leak:.1%}  ← 虚高!")
print("\n上线后 collection_calls 拿不到 (申请时还没催收) → 模型实际只有干净特征的水平")
print("→ 离线 98% 上线 88% — 经典的泄漏崩盘")


**目标泄漏的侦测法**：
1. **单特征预测力异常高** → 警报（一个特征就 0.95 AUC？太可疑）
2. **追问每个特征的"产生时间"**：它在预测时刻之前就存在吗？
3. **业务因果**：它是目标的因还是果？果 = 泄漏

经典真实案例：用"是否住院"预测"是否患病"（住院是患病的结果）、用"账户是否冻结"预测"是否欺诈"。0.3/3.1 节 Titanic 的 `alive ≡ survived` 也是目标泄漏。
Real cases: predicting illness from "was hospitalized", fraud from "account frozen". And Titanic's alive==survived.


<a id="3"></a>
## 3. 预处理泄漏：fit 在全数据 / Preprocessing Leakage

**最常见**（前几课反复警告）。在划分前对全数据 fit scaler/imputer/encoder/feature-selector → test 的统计信息渗入训练。


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline

# 极端演示: 特征选择泄漏 / extreme demo: feature-selection leakage
# 造 5000 个纯噪声特征, 目标完全随机 — 正确做法下应该 = 50% (瞎猜)
n2 = 200
X_noise = rng.normal(size=(n2, 5000))
y_rand = rng.integers(0, 2, n2)

# ❌ 错: 在全数据上选特征, 再 CV / WRONG: select features on ALL data first
selector = SelectKBest(f_classif, k=20).fit(X_noise, y_rand)   # 偷看了全部 y!
X_selected = selector.transform(X_noise)
acc_wrong = cross_val_score(LogisticRegression(max_iter=500), X_selected, y_rand, cv=5).mean()

# ✅ 对: 特征选择放进 pipeline, 每折只在 train 选 / RIGHT: selection inside pipeline
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=500))
acc_right = cross_val_score(pipe, X_noise, y_rand, cv=5).mean()

print(f"纯噪声 + 随机标签 (真实应 ≈ 50%):")
print(f"  ❌ 全数据选特征: CV = {acc_wrong:.1%}  ← 虚高! 选特征时偷看了 test 折的 y")
print(f"  ✅ pipeline 内选: CV = {acc_right:.1%}  ← 正确, 接近瞎猜")
print("\n5000 个噪声里总有 20 个'碰巧'和全部 y 相关 — 在全数据选就把这运气泄漏进了 CV")


**这个演示极具冲击力**：**纯随机噪声 + 随机标签**, 正确做法下只能 50%（瞎猜）, 但"全数据选特征"能虚高到 60-70%——**凭空造出了不存在的预测力**。这就是为什么**所有"看 y 的预处理"（特征选择、target 编码、过采样）都必须在 CV 折内做**。
Pure noise + random labels should give 50%, but whole-data feature selection inflates to 60-70% — conjuring predictive power from nothing.


<a id="4"></a>
## 4. 时间泄漏：用了未来 / Temporal Leakage

时序数据的专属陷阱。**用了"预测时刻之后"才知道的信息**。

| 泄漏 | 修复 |
|---|---|
| 随机划分时序数据（未来行进 train, 过去行进 test）| **按时间划分**（train 全在 test 之前）|
| 特征用了未来窗口（"未来 7 天均值"）| 只用**历史**窗口 |
| 用全期统计量标准化 | 只用**训练期**统计量 |


In [ ]:
# 演示: 时序数据随机划分 vs 时间划分 / random vs time-based split on time series
dates = pd.date_range("2025-01-01", periods=1000, freq="D")
trend = np.linspace(0, 10, 1000)                          # 上升趋势 / upward trend
ts = pd.DataFrame({
    "date": dates,
    "feature": trend + rng.normal(0, 1, 1000),
    "target": trend + rng.normal(0, 1, 1000),             # 和趋势同步 / follows trend
})

# ❌ 随机划分: test 的"未来"点和 train 的"过去"点混在一起 / random split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
tr_idx, te_idx = train_test_split(np.arange(1000), test_size=0.3, random_state=0)
lr = LinearRegression().fit(ts.feature.values[tr_idx].reshape(-1,1), ts.target.values[tr_idx])
r2_random = r2_score(ts.target.values[te_idx], lr.predict(ts.feature.values[te_idx].reshape(-1,1)))

# ✅ 时间划分: 前 70% train, 后 30% test / time-based split
split = 700
lr2 = LinearRegression().fit(ts.feature.values[:split].reshape(-1,1), ts.target.values[:split])
r2_time = r2_score(ts.target.values[split:], lr2.predict(ts.feature.values[split:].reshape(-1,1)))

print(f"随机划分:  test R² = {r2_random:.3f}  (虚高 — train 里有 test 之后的'未来'点)")
print(f"时间划分:  test R² = {r2_time:.3f}  (真实 — 只用过去预测未来)")
print("\n时序数据永远按时间划分! 随机划分让模型'见过未来', 上线时没有未来可见 (3.10 节细讲 TimeSeriesSplit)")


<a id="5"></a>
## 5. 分组泄漏：同一实体跨 train/test / Group Leakage

**同一个实体（病人/用户/设备）的多条记录被分到 train 和 test 两边**。模型记住了实体身份而非学到规律。


In [ ]:
# 模拟: 每个病人多次就诊, 同病人记录高度相似 / multiple visits per patient
n_patients = 200
visits_per = 5
patient_id = np.repeat(np.arange(n_patients), visits_per)
patient_effect = rng.normal(0, 3, n_patients)            # 每个病人的固有特质 / patient trait
X_med = patient_effect[patient_id] + rng.normal(0, 0.5, n_patients*visits_per)
y_med = (patient_effect[patient_id] + rng.normal(0, 0.5, n_patients*visits_per) > 0).astype(int)
Xm = X_med.reshape(-1, 1)

# ❌ 随机划分: 同一病人的就诊分散两边 / random split mixes a patient's visits
acc_random = cross_val_score(RandomForestClassifier(random_state=0), Xm, y_med, cv=5).mean()

# ✅ 按病人分组划分 / group-aware split (GroupKFold, 3.10 节正题)
from sklearn.model_selection import GroupKFold, cross_val_score as cvs
acc_group = cvs(RandomForestClassifier(random_state=0), Xm, y_med,
                cv=GroupKFold(5), groups=patient_id).mean()

print(f"随机划分 (同病人跨两边): CV = {acc_random:.1%}  ← 虚高")
print(f"按病人分组划分:          CV = {acc_group:.1%}  ← 真实")
print("\n随机划分时, test 的某次就诊在 train 里有'同一病人的另一次就诊'(极相似)")
print("→ 模型靠'认出病人'作弊, 而非学到医学规律. 真实新病人来了就废")
print("→ 凡是有重复实体 (用户/病人/设备) 的数据, 必须 GroupKFold (3.10)")


<a id="6"></a>
## 6. 重复样本泄漏 / Duplicate Leakage

**简单但常见**：数据有重复行, **去重前就划分** → 同一行同时在 train 和 test → test 上"预测"其实是"背答案"。

防御：**划分前先去重**（或至少检查重复率）。爬虫数据、日志数据尤其要小心。


In [ ]:
# 演示 / demo
base = pd.DataFrame({"x": rng.normal(size=(100, 1)).ravel(), "y": rng.integers(0,2,100)})
# 人为复制 50% 的行 (模拟数据收集时的重复) / duplicate 50% of rows
dup = pd.concat([base, base.sample(50, random_state=1)], ignore_index=True)

# ❌ 不去重就划分 / split without dedup
X_tr, X_te, y_tr, y_te = train_test_split(dup[["x"]], dup["y"], test_size=0.3, random_state=0)
overlap = pd.merge(X_tr.assign(s="tr"), X_te.assign(s="te"), on="x", how="inner")
print(f"含重复数据直接划分: train 和 test 有 {len(overlap)} 个完全相同的 x 值")
print("→ 这些'test'样本在 train 里有副本, 模型见过, test 分数虚高")
print(f"\n防御: 先 df.drop_duplicates() (原 {len(dup)} 行 → 去重 {len(dup.drop_duplicates())} 行) 再划分")


<a id="7"></a>
## 7. 泄漏侦测清单 ⭐ / The Leakage Detection Checklist

```
🚨 警报信号:
  □ 离线分数好得不真实 (CV 0.99 / AUC 0.999) → 先怀疑泄漏, 别庆祝
  □ 某单一特征预测力异常高 → 查它是不是目标的"果"
  □ 离线分数 >> 生产分数 → 泄漏 (vs 过拟合: 过拟合是 train>>test, 泄漏是 test 也虚高)

🔍 逐项排查:
  □ 每个特征: 预测时刻拿得到吗? 是目标的因还是果?
  □ 所有 fit (scaler/imputer/encoder/selector): 只在 train 吗? → 用 Pipeline 保证
  □ 时序数据: 按时间划分了吗? 特征用了未来窗口吗?
  □ 有重复实体 (用户/病人): 用 GroupKFold 了吗?
  □ 划分前去重了吗?
  □ target 编码/过采样: 在 CV 折内做的吗?

🛡️ 结构性防御:
  □ 一切预处理放进 sklearn Pipeline (3.12)
  □ 用对的 CV 划分 (3.10): 时序→TimeSeriesSplit, 分组→GroupKFold
  □ 留一个"上线前从未碰过"的 holdout 集做最终验收
```

> 💡 **泄漏 vs 过拟合的诊断**（面试常混）：
> - **过拟合**: train 高, test/CV 低 → 模型记住了训练噪声 → 正则化/简化模型
> - **泄漏**: train 高, **test/CV 也高**, 但**生产低** → 信息穿越 → 查预处理/特征/划分
> Overfitting: high train, low test. Leakage: high train AND test, low production.


<a id="8"></a>
## 8. 小结 / Summary

```
泄漏本质: 预测时刻拿不到的信息进了训练 → 离线虚高, 生产崩盘
五种形态:
  目标泄漏 (特征是目标的果) — 最隐蔽, 追问因果
  预处理泄漏 (fit 全数据) — 最常见, 用 Pipeline
  时间泄漏 (用未来) — 时序按时间划分
  分组泄漏 (同实体跨两边) — GroupKFold
  重复泄漏 (没去重就划分) — 先去重
震撼演示: 纯噪声+随机标签, 全数据选特征能虚高到 60-70% (真实 50%)
诊断: 离线虚高=警报; 泄漏(test也高生产低) vs 过拟合(test就低)
```

### 💡 面试速查
1. **泄漏一句话**: 模型偷看了预测时刻不可能有的信息
2. **目标泄漏**: 特征是目标的果不是因 (退款→欺诈, 住院→患病)
3. **预处理泄漏**: 所有 fit 只在 train (Pipeline 保证)
4. **泄漏 vs 过拟合**: 泄漏 test 也虚高, 过拟合 test 就低
5. **结构性防御**: Pipeline + 正确 CV + holdout

### 下一节
**3.10 划分与交叉验证**——本课反复说"按时间/分组划分", 下一课系统讲：hold-out / K-fold / 分层 / GroupKFold / TimeSeriesSplit。
